# Telecom Customer Churn — 05. Data Cleaning & Preparation

**Goal:** Convert the raw dataset into a validated, analysis-ready dataset while preserving the original information and documenting every meaningful transformation.

> The raw dataset is never overwritten.

## 1. Imports, paths, and reproducibility

We use `Path` for safer file-path handling on Windows and other systems.

The raw file remains the source of truth; the cleaned file is written to a separate location.

In [ ]:
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path(r"D:\Data Analytics\Project\2) Dataset\1) Raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

CLEANED_DIR = Path(r"D:\Data Analytics\Project\2) Dataset\2) Cleaned")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_DATA_PATH = CLEANED_DIR / "telco_churn_cleaned.csv"

## 2. Load the raw dataset and create a working copy

We use `raw_df` as the untouched source and `clean_df` as the working dataset.

This separation makes the workflow reproducible and prevents accidental modification of the raw data.

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)
clean_df = raw_df.copy()

print("Raw shape:", raw_df.shape)
print("Working shape:", clean_df.shape)

## 3. Standardize column names

We convert field names to a consistent `snake_case` convention.

This fixes an important problem in the original snippets: after renaming columns, code must use the **new names consistently**.

For example:
- `CustomerID` → `customer_id`
- `MonthlyCharges` → `monthly_charges`
- `TotalCharges` → `total_charges`

In [ ]:
clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

clean_df.columns.tolist()

## 4. Standardize surrounding whitespace in text fields

We remove accidental leading/trailing whitespace from object/string columns.

We do not change meaningful category wording such as `No internet service`.

In [ ]:
text_cols = clean_df.select_dtypes(include="object").columns

for col in text_cols:
    clean_df[col] = clean_df[col].str.strip()

## 5. Validate the standardized categorical values

The common Telco dataset already uses consistent category labels. We inspect the values after trimming rather than applying an unnecessary mapping.

This is an improvement over the original `contract_map`, which could accidentally change valid data if the source uses different labels.

In [ ]:
categorical_cols = [
    "gender",
    "partner",
    "dependents",
    "phoneservice",
    "multiplelines",
    "internetservice",
    "contract",
    "paperlessbilling",
    "paymentmethod",
    "churn",
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(clean_df[col].value_counts(dropna=False))

## 6. Convert numeric columns explicitly

`tenure` and `monthly_charges` should be numeric.

`total_charges` needs special handling because blank strings in the raw dataset can prevent direct numeric interpretation.

In [ ]:
numeric_cols = [
    "tenure",
    "monthly_charges",
    "total_charges",
]

for col in numeric_cols:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

clean_df[numeric_cols].dtypes

## 7. Inspect `total_charges` missingness after conversion

This is the point where the raw blank strings become `NaN`.

We inspect the affected records before deciding how to treat them.

In [ ]:
total_charges_missing = clean_df["total_charges"].isna()

print("Missing total_charges:", int(total_charges_missing.sum()))

clean_df.loc[
    total_charges_missing,
    [
        "customer_id",
        "tenure",
        "monthly_charges",
        "contract",
        "churn",
    ]
]

## 8. Apply the documented `TotalCharges` treatment

For the standard Telco Customer Churn dataset, the problematic `TotalCharges` records are associated with zero tenure. In that specific situation, treating accumulated total charges as `0` is a defensible business rule.

We still verify the condition before filling.

If any missing record does **not** have zero tenure, we stop and investigate instead of silently filling it.

In [ ]:
unexpected_total_charge_missing = clean_df[
    clean_df["total_charges"].isna() &
    clean_df["tenure"].ne(0)
]

if not unexpected_total_charge_missing.empty:
    raise ValueError(
        "Some missing TotalCharges records do not have zero tenure. "
        "Investigate them before applying the zero-charge rule."
    )

clean_df["total_charges"] = clean_df["total_charges"].fillna(0)

print(
    "Remaining missing total_charges:",
    int(clean_df["total_charges"].isna().sum())
)

## 9. Check duplicate rows before removing anything

We only remove exact duplicate rows after confirming that they are genuine duplicates.

If the count is zero, no action is needed.

In [ ]:
duplicate_rows = clean_df.duplicated().sum()
print("Exact duplicate rows:", duplicate_rows)

if duplicate_rows > 0:
    clean_df = clean_df.drop_duplicates().copy()

print("Shape after duplicate-row handling:", clean_df.shape)

## 10. Validate customer-ID uniqueness

The dataset is expected to have one row per customer.

If duplicate customer IDs exist, we inspect them rather than dropping rows automatically.

In [ ]:
duplicate_customer_mask = clean_df["customer_id"].duplicated(keep=False)

print("Missing customer IDs:", clean_df["customer_id"].isna().sum())
print("Unique customer IDs:", clean_df["customer_id"].nunique())
print("Rows:", len(clean_df))
print("Duplicate customer IDs:", clean_df["customer_id"].duplicated().sum())

if duplicate_customer_mask.any():
    display(
        clean_df.loc[duplicate_customer_mask]
        .sort_values("customer_id")
    )

## 11. Validate numeric ranges

These checks confirm that the cleaned numeric fields contain plausible non-negative values.

We do not remove outliers here simply because they are large.

In [ ]:
negative_counts = {
    "tenure": int((clean_df["tenure"] < 0).sum()),
    "monthly_charges": int((clean_df["monthly_charges"] < 0).sum()),
    "total_charges": int((clean_df["total_charges"] < 0).sum()),
}

negative_counts

## 12. Stop if impossible negative values exist

If any negative value is found, we should investigate the source rather than silently converting it.

For the standard dataset, these checks should normally return zero.

In [ ]:
if any(negative_counts.values()):
    raise ValueError(
        f"Impossible negative values detected: {negative_counts}. "
        "Investigate before continuing."
    )

## 13. Create a readable senior-citizen label

The original `seniorcitizen` field is encoded as `0/1`.

We retain the original numeric field and add a business-friendly label.

In [ ]:
clean_df["senior_citizen_label"] = (
    clean_df["seniorcitizen"]
    .map({0: "No", 1: "Yes"})
)

if clean_df["senior_citizen_label"].isna().any():
    raise ValueError("Unexpected SeniorCitizen values found.")

## 14. Create a numeric churn flag

We keep the original `churn` text field for readability and add `churn_flag` for calculations.

- `Yes` → 1
- `No` → 0

In [ ]:
unexpected_churn = set(clean_df["churn"].dropna().unique()) - {"Yes", "No"}

if unexpected_churn:
    raise ValueError(f"Unexpected churn values: {unexpected_churn}")

clean_df["churn_flag"] = (
    clean_df["churn"] == "Yes"
).astype(int)

## 15. Create tenure groups

The original `tenure` variable is preserved.

`tenure_group` is a derived business-friendly segmentation used for churn comparisons.

In [ ]:
tenure_bins = [0, 12, 24, 48, float("inf")]
tenure_labels = [
    "0–12 months",
    "13–24 months",
    "25–48 months",
    "49+ months",
]

clean_df["tenure_group"] = pd.cut(
    clean_df["tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    include_lowest=True,
    right=True,
)

print(clean_df["tenure_group"].value_counts(sort=False))

## 16. Create monthly-charge bands

This is an analytical preparation field, not a business conclusion.

We use tertiles so that the groups are approximately balanced in size. `duplicates="drop"` prevents a technical failure if identical quantile boundaries occur.

In [ ]:
clean_df["monthly_charge_band"] = pd.qcut(
    clean_df["monthly_charges"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop",
)

print(clean_df["monthly_charge_band"].value_counts(sort=False))

## 17. Create a high-monthly-charge flag

We use the 75th percentile as an initial analytical flag.

This does **not** mean that these customers are automatically high-risk. It simply creates a reusable segment for later investigation.

In [ ]:
high_charge_threshold = clean_df["monthly_charges"].quantile(0.75)

clean_df["high_monthly_charge"] = (
    clean_df["monthly_charges"] >= high_charge_threshold
)

print("75th-percentile threshold:", high_charge_threshold)

## 18. Validate the cleaned dataset

We repeat the main quality checks after cleaning.

This is essential: a cleaning operation can introduce new problems, so the result must be audited again.

In [ ]:
print("=== CLEAN DATA VALIDATION ===")
print("Shape:", clean_df.shape)
print("Missing cells:", int(clean_df.isna().sum().sum()))
print("Duplicate rows:", int(clean_df.duplicated().sum()))
print("Duplicate customer IDs:", int(clean_df["customer_id"].duplicated().sum()))

print("\nMissing values by column:")
print(clean_df.isna().sum()[clean_df.isna().sum() > 0])

print("\nData types:")
print(clean_df.dtypes)

## 19. Review the final numeric summary

This confirms that the numeric columns are now genuinely numeric and provides a final sanity check on their distributions.

In [ ]:
clean_df[
    ["seniorcitizen", "tenure", "monthly_charges", "total_charges", "churn_flag"]
].describe()

## 20. Save the cleaned dataset

We export a separate CSV file.

The raw source file remains untouched.

In [ ]:
clean_df.to_csv(
    CLEANED_DATA_PATH,
    index=False
)

print(f"Saved cleaned dataset to: {CLEANED_DATA_PATH}")